# 04 - Neural Network Model

**Project:** Predictive Modeling for Drug Discovery via Virtual Screening  
**Student:** Milica Jeftic (ID: 89211255)  
**Date:** January 2026  
**Dataset:** Kaggle - Drug Discovery Virtual Screening Dataset

---

## Goal of This Notebook

This notebook implements a feed-forward neural network for the compound activity classification task. This model family was included in the project proposal to test whether a deeper model can learn useful non-linear relationships between molecular/protein descriptors and biological activity.

The neural network is compared with the Logistic Regression baseline and tree-based models using the same train/validation/test split prepared in notebook 01.

---

## Expected Outputs

- Neural network validation and test metrics
- Training and validation learning curves
- Confusion matrix and ROC curve visualizations
- Saved Keras model in `models/`
- Saved metrics table in `results/metrics/`


## 1. Environment Setup

This section imports the required libraries, configures reproducibility, and defines project paths.

In [ ]:
# ============================
# Environment & Configuration
# ============================

import os
import sys
import random
import warnings

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# ----------------------------
# Reproducibility & Warnings
# ----------------------------
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ----------------------------
# Pandas display options
# ----------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

# ----------------------------
# Visualization defaults
# ----------------------------
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
rcParams["figure.figsize"] = (12, 6)
rcParams["font.size"] = 12

%matplotlib inline

# ----------------------------
# Project paths
# ----------------------------
def find_project_root(start_path):
    """Find the project root from either the repository root or the notebooks directory."""
    current_path = os.path.abspath(start_path)
    for _ in range(3):
        expected_items = [
            os.path.join(current_path, "data"),
            os.path.join(current_path, "notebooks"),
            os.path.join(current_path, "README.md"),
        ]
        if all(os.path.exists(path) for path in expected_items):
            return current_path
        current_path = os.path.dirname(current_path)
    raise FileNotFoundError("Could not locate the project root directory.")

PROJECT_ROOT = find_project_root(os.getcwd())
DATA_PROCESSED_PATH = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_PATH = os.path.join(PROJECT_ROOT, "models")
RESULTS_PATH = os.path.join(PROJECT_ROOT, "results")

print("=" * 60)
print("Environment initialized successfully")
print("=" * 60)
print(f"Python       : {sys.version.split()[0]}")
print(f"Numpy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"TensorFlow   : {tf.__version__}")
print(f"Scikit-learn : {__import__('sklearn').__version__}")
print("-" * 60)
print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_PROCESSED_PATH}")
print(f"Models dir   : {MODELS_PATH}")
print("=" * 60)


## 2. Load Preprocessed Data

The neural network uses the same scaled train, validation, and test sets produced in notebook 01. No additional preprocessing is performed here.

In [ ]:
print("=" * 60)
print("LOADING PREPROCESSED DATA")
print("=" * 60)

X_train = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "X_train.csv"))
X_val = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "X_val.csv"))
X_test = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "X_test.csv"))

y_train = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "y_train.csv")).squeeze("columns").astype(int)
y_val = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "y_val.csv")).squeeze("columns").astype(int)
y_test = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "y_test.csv")).squeeze("columns").astype(int)

assert len(X_train) == len(y_train), "Train X/y length mismatch"
assert len(X_val) == len(y_val), "Validation X/y length mismatch"
assert len(X_test) == len(y_test), "Test X/y length mismatch"
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns), "Feature columns do not match across splits"

feature_cols = list(X_train.columns)
input_dim = X_train.shape[1]

print("Data loaded successfully")
print(f"Training set   : X={X_train.shape}, y={y_train.shape}")
print(f"Validation set : X={X_val.shape}, y={y_val.shape}")
print(f"Test set       : X={X_test.shape}, y={y_test.shape}")
print(f"Input dimension: {input_dim}")

print("\nClass proportions:")
for split_name, split_y in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{split_name}: {split_y.value_counts(normalize=True).sort_index().to_dict()}")


## 3. Model Definition

The neural network is a small fully connected binary classifier. Dropout and L2 regularization are included to reduce overfitting, and early stopping is used to stop training when validation loss stops improving.

In [ ]:
def build_neural_network(input_dim):
    """Build a regularized feed-forward neural network for binary classification."""
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(
            32,
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001),
        ),
        layers.Dropout(0.20),
        layers.Dense(
            16,
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001),
        ),
        layers.Dropout(0.10),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )
    return model

nn_model = build_neural_network(input_dim)
nn_model.summary()


## 4. Training

Class weights are computed from the training labels to account for the moderate 70/30 class imbalance. The model is trained on the training set and monitored on the validation set.

In [ ]:
print("=" * 60)
print("TRAINING NEURAL NETWORK")
print("=" * 60)

classes = np.array([0, 1])
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weights = {int(cls): float(weight) for cls, weight in zip(classes, class_weights_array)}

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
    )
]

history = nn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=0,
)

print("Training completed")
print(f"Epochs run: {len(history.history['loss'])}")
print(f"Class weights: {class_weights}")
print(f"Best validation loss: {min(history.history['val_loss']):.4f}")


## 5. Learning Curves

Learning curves are used to inspect convergence and possible overfitting.

In [ ]:
figures_path = os.path.join(RESULTS_PATH, "figures")
os.makedirs(figures_path, exist_ok=True)

history_df = pd.DataFrame(history.history)

epochs = np.arange(1, len(history_df) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history_df["loss"], label="Training loss")
axes[0].plot(epochs, history_df["val_loss"], label="Validation loss")
axes[0].set_title("Neural Network Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary cross-entropy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history_df["accuracy"], label="Training accuracy")
axes[1].plot(epochs, history_df["val_accuracy"], label="Validation accuracy")
axes[1].set_title("Neural Network Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
learning_curve_path = os.path.join(figures_path, "neural_network_learning_curves.png")
plt.savefig(learning_curve_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Learning curves saved to: {learning_curve_path}")


## 6. Validation and Test Evaluation

The neural network is evaluated using the same metrics as the previous notebooks: accuracy, precision, recall, F1-score, and ROC-AUC.

In [ ]:
def evaluate_neural_network(model, X, y, split_name):
    """Evaluate the neural network and return metrics plus predictions."""
    y_proba = model.predict(X, verbose=0).ravel()
    y_pred = (y_proba >= 0.5).astype(int)

    metrics = {
        "model": "Neural Network",
        "split": split_name,
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred),
        "recall": recall_score(y, y_pred),
        "f1_score": f1_score(y, y_pred),
        "roc_auc": roc_auc_score(y, y_proba),
    }
    return metrics, y_pred, y_proba

val_metrics, y_val_pred, y_val_proba = evaluate_neural_network(nn_model, X_val, y_val, "validation")
test_metrics, y_test_pred, y_test_proba = evaluate_neural_network(nn_model, X_test, y_test, "test")

nn_metrics_df = pd.DataFrame([val_metrics, test_metrics])
display(nn_metrics_df)


## 7. Visual Evaluation

The confusion matrix and ROC curve summarize neural network performance on the held-out test set.

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
test_auc = roc_auc_score(y_test, y_test_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    ax=axes[0],
    cbar=False,
    xticklabels=["Inactive", "Active"],
    yticklabels=["Inactive", "Active"],
)
axes[0].set_title("Confusion Matrix - Neural Network Test Set", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

axes[1].plot(fpr, tpr, linewidth=2, label=f"ROC-AUC = {test_auc:.4f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Random Classifier")
axes[1].set_title("ROC Curve - Neural Network Test Set", fontsize=12, fontweight="bold")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
test_plot_path = os.path.join(figures_path, "neural_network_test_evaluation.png")
plt.savefig(test_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Neural network test visualization saved to: {test_plot_path}")


## 8. Save Model and Metrics

The trained neural network and metrics table are saved for final comparison in notebook 05.

In [ ]:
print("=" * 60)
print("SAVING NEURAL NETWORK OUTPUTS")
print("=" * 60)

os.makedirs(MODELS_PATH, exist_ok=True)
metrics_path = os.path.join(RESULTS_PATH, "metrics")
os.makedirs(metrics_path, exist_ok=True)

nn_model_path = os.path.join(MODELS_PATH, "neural_network.keras")
nn_metrics_path = os.path.join(metrics_path, "neural_network_metrics.csv")

nn_model.save(nn_model_path)
nn_metrics_df.to_csv(nn_metrics_path, index=False)

print(f"Neural network model saved to: {nn_model_path}")
print(f"Neural network metrics saved to: {nn_metrics_path}")
display(nn_metrics_df)


## 9. Neural Network Summary

This notebook trained a regularized feed-forward neural network as the deep learning model proposed for the project. It uses the same processed train/validation/test split as the previous models, applies class weighting for the 70/30 imbalance, and uses early stopping to reduce overfitting.

### Key Results

The neural network achieved strong performance on both validation and test sets:

- Validation accuracy: 0.9891
- Validation F1-score: 0.9822
- Validation ROC-AUC: 0.9997
- Test accuracy: 0.9927
- Test F1-score: 0.9880
- Test ROC-AUC: 0.9999

These results improve slightly over the Logistic Regression baseline, but they do not exceed the tree-based models from notebook 03, which achieved perfect scores on this dataset.

### Interpretation

The neural network confirms that the processed descriptors contain enough signal to predict activity very accurately. However, because the tree-based models already reached perfect performance and feature importance showed that `binding_affinity` is highly dominant, the neural network should be interpreted as an additional model-family comparison rather than a clear practical improvement.

The saved neural network model and metrics will be included in the final model comparison notebook.
